# Audiobook Workflow

This notebook separates story generation from audio generation. First, it creates detailed story text from the PRD and prompt. After review, it generates a short preview audio and then a full audiobook audio file.

In [2]:
import sys
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Audio, display
from pprint import pprint
import importlib

root_dir = Path.cwd().parent
sys.path.append(str(root_dir))

import whisper_client
importlib.reload(whisper_client)
from whisper_client import generate_audiobook_audio, convert_story_to_audio

load_dotenv()

project_root = root_dir
prd_file = project_root / 'audiobook_prd.md'
preview_audio_file = project_root / 'output' / 'sample_story_preview_audio_india_trip.mp3'
full_audio_file = project_root / 'output' / 'full_story_india_trip_audio.mp3'

print(f'Using PRD file: {prd_file}')
print(f'Preview audio file: {preview_audio_file}')
print(f'Full audio file: {full_audio_file}')

Using PRD file: /Users/parikshitmehta/Documents/audiobook_project/audiobook_prd.md
Preview audio file: /Users/parikshitmehta/Documents/audiobook_project/output/sample_story_preview_audio_india_trip.mp3
Full audio file: /Users/parikshitmehta/Documents/audiobook_project/output/full_story_india_trip_audio.mp3


## Story prompt

Define the specific plot and story elements for the audiobook. This prompt will drive generation of the narrative text without creating any audio yet.

In [ ]:
story_prompt = (
""" Meera is going on a vecation trip to Myrtle Beach, SC with her family. In this trip, she is excited about building sandcastles, collecting seashells, and splashing in the waves. She loves the sound of the ocean and the feeling of the warm sun on her skin. 
Meera loves to spend time in the pool with her dad, playing games and having fun. She loves building sandcastle with her mom.
After spending all morning at the beach and taking a little nap in the afternoon, Meera goes to a shopping place and has some ice cream.
Create the story to include more fun escapades that are appropriate for a 4 year old. Use warm, calm narration and simple language appropriate for a 4-year-old.

"""
)

story_prompt = ( """ 
It's December and the Mehta family is going on a trip to India to visit their relatives. Meera is excited to see her grandparents, aunts, uncles, and cousins. She loves playing with her cousins and listening to her grandparents' stories about when they were young.
During the trip, Meera enjoys eating delicious Indian food like samosas, parathas, and hakka noodles. She also loves going to the park with her cousins and playing games. One day, Meera and her family go to a big market where they see colorful clothes, toys, and jewelry. Meera gets a new dress and some bangles that she loves.
Meera takes a train ride from Amdavad to Pune to visit her aunt Minu masi. She wants to see her cousing Anant. Once there, Anant takes meera to a big zoo. Meera loves seeing all the animals, especially the elephants and monkeys. She also enjoys going to the park with Anant and playing on the swings and slides.
After coming back to Amddavad, Meera's cousins  Sidh and Sara take her to a science museum where she learns about planets, stars, and dinosaurs. Meera loves the dinosaur exhibit and pretends to be a dinosaur roaring and stomping around. Meera spends time with her cousins Malhar and Krishnavi at Nana Nani's place. Where they have a lot of fun playing in the house as well as going to the mall. 
After spending 3 weeks , Meera is sad to leave her family in India but is happy to have had such a fun trip. She can't wait to tell her friends at school all about her adventures in India and show them the pictures she took with her cousins and grandparents. Meera is grateful for the time she spent with her family and the memories they created together. She knows that she will always cherish this trip to India in her heart.
Create a story appropriate for a 4 year old based on the above details. Use warm, calm narration and simple language appropriate for a 4-year-old.                                                                                   


""")




print(story_prompt)

 
It's December and the Mehta family is going on a trip to India to visit their relatives. Meera is excited to see her grandparents, aunts, uncles, and cousins. She loves playing with her cousins and listening to her grandparents' stories about when they were young.
During the trip, Meera enjoys eating delicious Indian food like samosas, dosas, and hakka noodles. She also loves going to the park with her cousins and playing games. One day, Meera and her family go to a big market where they see colorful clothes, toys, and jewelry. Meera gets a new dress and some bangles that she loves.
Meera takes a train ride from Amdavad to Pune to visit her aunt Minu masi. She wants to see her cousing Anant. Once there, Anant takes meera to a big zoo. Meera loves seeing all the animals, especially the elephants and monkeys. She also enjoys going to the park with Anant and playing on the swings and slides.
After coming back to Amddavad, Meera's cousins  Sidh and Sara take her to a science museum where

## Generate story text only

Run this cell to generate the full story text based on the PRD and prompt. No audio is created at this stage. Review the text output before generating any audio.

In [ ]:
story_result = generate_audiobook_audio(
    prd_file_path=str(prd_file),
    prompt=story_prompt,
    output_file_path=None,
    generate_audio=False,
    story_model='gpt-5-2',
)

story_text = story_result['story_text']
print('--- Generated story text ---')
pprint(story_text)

--- Generated story text ---
('{"id":"resp_0b9c7e8fcd767e44006a18ed53f50c8191b0bc7d627949e8ee","created_at":1780018515.0,"error":null,"incomplete_details":null,"instructions":null,"metadata":{},"model":"gpt-4o-mini-2024-07-18","object":"response","output":[{"id":"msg_0b9c7e8fcd767e44006a18ed54a3588191811507a5312a908b","content":[{"annotations":[],"text":"Once '
 'upon a time, in the cozy month of December, the Mehta family was getting '
 'ready for a special trip. They were going all the way to India to visit '
 'their family. Little Meera was so excited! She couldn’t wait to see her '
 'grandparents, aunts, uncles, and her playful cousins.\\n\\nAs soon as they '
 'arrived in India, the warm sun greeted them. Meera rushed into her '
 'grandparents’ big, happy home. She hugged her grandmother tight and smiled '
 'at her grandfather, who had the funniest stories to tell. Meera loved '
 'listening to tales about when her grandparents were little, just like '
 'her.\\n\\nEvery day was fill

## Review and approve the story text

Read the generated story text above. Once the narrative looks good, run the next cell to create a short audio preview from the beginning of the story.

In [15]:
preview_sentences = story_text.split('.')[:30]
preview_text = '.'.join(s.strip() for s in preview_sentences if s).strip()
if not preview_text.endswith('.'):
    preview_text += '.'

print('--- Preview text to be converted to audio ---')
pprint(preview_text)

preview_audio_path = convert_story_to_audio(
    story_text=preview_text,
    output_file_path=str(preview_audio_file),
    tts_model='gpt-4o-mini-tts',
    voice='marin',
)

print(f'Generated preview audio: {preview_audio_path}')
display(Audio(filename=str(preview_audio_path)))

--- Preview text to be converted to audio ---
('{"id":"resp_0b9c7e8fcd767e44006a18ed53f50c8191b0bc7d627949e8ee","created_at":1780018515.0,"error":null,"incomplete_details":null,"instructions":null,"metadata":{},"model":"gpt-4o-mini-2024-07-18","object":"response","output":[{"id":"msg_0b9c7e8fcd767e44006a18ed54a3588191811507a5312a908b","content":[{"annotations":[],"text":"Once '
 'upon a time, in the cozy month of December, the Mehta family was getting '
 'ready for a special trip.They were going all the way to India to visit their '
 'family.Little Meera was so excited! She couldn’t wait to see her '
 'grandparents, aunts, uncles, and her playful cousins.\\n\\nAs soon as they '
 'arrived in India, the warm sun greeted them.Meera rushed into her '
 'grandparents’ big, happy home.She hugged her grandmother tight and smiled at '
 'her grandfather, who had the funniest stories to tell.Meera loved listening '
 'to tales about when her grandparents were little, just like her.\\n\\nEvery '
 '

## Review the preview audio

Listen to the short preview and verify the narration. If it sounds correct, run the final cell to generate the full audiobook audio from the full story text.

In [16]:
full_audio_path = convert_story_to_audio(
    story_text=story_text,
    output_file_path=str(full_audio_file),
    tts_model='gpt-4o-mini-tts',
    voice='marin',
)

print(f'Generated full story audio: {full_audio_path}')
display(Audio(filename=str(full_audio_path)))

Generated full story audio: full_story_india_trip_audio.mp3
